# ***Search Engine With Tools And Agents***

### ***This section introduces the concept of building a search engine using various tools and agents.***

In [1]:
# Arxiv--Research
# Tools creation
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [42]:
# Used the inbuilt tool of Wikipedia
# Initialize Wikipedia API wrapper with specific parameters
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)

In [ ]:
# Display the Wikipedia API wrapper object
api_wrapper_wiki

WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\emon1\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)

In [44]:
# Create a Wikipedia query run tool using the API wrapper
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)

In [45]:
# Display the name of the Wikipedia tool
wiki.name

'wikipedia'

In [ ]:
# Initialize Arxiv API wrapper with specific parameters
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=250)

In [46]:
# Create an Arxiv query run tool using the API wrapper
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)

arxiv


In [47]:
# Combine the tools into a list
tools = [wiki, arxiv]

In [48]:
# Custom tools [RAG Tool]
# Import necessary libraries for creating custom tools
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings  # Use Hugging Face Embeddings as an alternative to OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [49]:
# Load documents from a web source
loader = WebBaseLoader("https://docs.smith.langchain.com/")

In [50]:
# Load the documents
docs = loader.load()

In [51]:
# Split the documents into chunks for processing
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)

In [52]:
# Create a vector database using FAISS and Hugging Face embeddings
vectordb = FAISS.from_documents(documents, HuggingFaceEmbeddings())

In [53]:
# Create a retriever from the vector database
retriever = vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020417A497D0>, search_kwargs={})

In [54]:
# Import the function to create a retriever tool
from langchain.tools.retriever import create_retriever_tool

In [55]:
# Create a retriever tool for searching Langsmith information
retriever_tool = create_retriever_tool(retriever, "langsmith-search", "Search any information about Langsmith")

In [56]:
# Display the name of the retriever tool
retriever_tool.name

'langsmith-search'

In [57]:
# Combine all tools into a single list
tools = [wiki, arxiv, retriever_tool]

In [58]:
# Display the list of tools
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\emon1\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 Tool(name='langsmith-search', description='Search any information about Langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x0000020448A3E200>, retriever=VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS ob

In [59]:
# Tools, LLM --> AgentExecutor
# Import necessary libraries for using Groq model and environment variables
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

In [60]:
# Load environment variables from .env file
load_dotenv()

True

In [61]:
# Retrieve the Groq API key from environment variables
groq_api_key = os.getenv("GROQ_API_KEY")

In [62]:
# Using the Groq model for language tasks
# Initialize the Groq model with specific parameters
llm = ChatGroq(groq_api_key=groq_api_key, model_name="Llama3-8b-8192")

In [63]:
# Prompt Template
# Import the hub for pulling prompt templates
from langchain import hub

In [64]:
# Pull a specific prompt template from the hub
prompt = hub.pull("hwchase17/openai-functions-agent")  # Update the prompt as per your task

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\langsmith\client.py:253: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [65]:
# Display the messages in the prompt template
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [66]:
# Agents
# Import the function to create an OpenAI tools agent
from langchain.agents import create_openai_tools_agent

In [67]:
# Create the agent using the Groq model and the tools
agent = create_openai_tools_agent(llm, tools, prompt)

In [68]:
# Display the agent object
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

In [69]:
# Agent Executor
# Import the AgentExecutor class
from langchain.agents import AgentExecutor

In [70]:
# Initialize the AgentExecutor with the agent and tools
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [71]:
# Display the AgentExecutor object
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

In [72]:
# Test the agent with the provided queries
response1 = agent_executor.invoke({"input": "Tell me about Langsmith"})
response2 = agent_executor.invoke({"input": "What is machine learning"})
response3 = agent_executor.invoke({"input": "What's the paper 1706.03762 about?"})



> Entering new AgentExecutor chain...

Invoking: `langsmith-search` with `{'query': 'Langsmith'}`


Get started with LangSmith | ü¶úÔ∏èüõ†Ô∏è LangSmith

LangSmith + LangChain OSSLangSmith integrates seamlessly with LangChain's open source frameworks langchain and langgraph, with no extra instrumentation needed.If you're already using either of these, see the how-to guide for setting up LangSmith with LangChain or setting up LangSmith with LangGraph.
Observability‚Äã
Observability is important for any software application, but especially so for LLM applications. LLMs are non-deterministic by nature, meaning they can produce unexpected results. This makes them trickier than normal to debug.
This is where LangSmith can help! LangSmith has LLM-native observability, allowing you to get meaningful insights from your application. LangSmith‚Äôs observability features have you covered throughout all stages of application development - from prototyping, to beta testing, to production.

Skip 

In [41]:
# Show responses
print(response1)
print(response2)
print(response3)

{'input': 'Tell me about Langsmith', 'output': "It seems like LangSmith is a platform for building production-grade Large Language Model (LLM) applications. It provides features for observability, evaluation, prompt engineering, and deployment. With LangSmith, you can closely monitor and evaluate your application, iterate on prompts, and get human feedback on your data.\n\nIt seems like LangSmith integrates with LangChain's open-source frameworks langchain and langgraph, and it provides a set of tools for prompt engineering to help you find the perfect prompt for your application.\n\nLangSmith also provides features for observability, such as tracing, metrics, dashboards, and alerts, which can help you analyze your application's performance and get meaningful insights. Additionally, it provides a set of evaluators for evaluating your application's performance and collecting human feedback on your data.\n\nIt seems like LangSmith is a comprehensive platform for building and deploying LL